
# Build **Optimal Delegation** States (per epoch)

Assign, for each epoch, each delegator to the **closest** DRep by opinion (ties by `drep_id`).

**Inputs (`csv_out/`):**
- `dreps_state.csv` — `epoch, drep_id, opinion, stake`
- `delegators_state.csv` — `epoch, delegator_id, opinion, stake, s`

**Outputs (`csv_out_optimal/`):**
1. `delegators_state_with_delegations.csv` — `epoch, delegator_id, opinion, stake, s, drep_id, drep_opinion, distance`
2. `dreps_state_with_wprime.csv` — `epoch, drep_id, opinion, stake, delegated_stake, indegree, avg_distance, Wprime, Wprime_share`


In [14]:

import pandas as pd
from pathlib import Path

IN_DIR = Path("csv_out_exp")
OUT_DIR = Path("csv_out_optimal")
OUT_DIR.mkdir(parents=True, exist_ok=True)

dreps = pd.read_csv(IN_DIR / "dreps_state.csv")
deleg = pd.read_csv(IN_DIR / "delegators_state.csv")

assert set(['epoch','drep_id','opinion','stake']).issubset(dreps.columns)
assert set(['epoch','delegator_id','opinion','stake','s']).issubset(deleg.columns)

dreps.head(), deleg.head()

(   epoch drep_id   opinion     stake
 0      0      d1  0.386217  0.488469
 1      0      d2  0.450241  0.377078
 2      0      d3  0.652259  0.627428
 3      0      d4  0.644337  0.641185
 4      0      d5  0.161554  0.544423,
    epoch delegator_id   opinion     stake         s
 0      0           a1  0.802161  0.590105  0.057942
 1      0           a2  0.694992  0.884621  0.656573
 2      0           a3  0.269281  0.027263  0.469737
 3      0           a4  0.708820  0.196704  0.480557
 4      0           a5  0.635183  0.025914  0.906085)

In [15]:

def assign_closest(Ae: pd.DataFrame, De: pd.DataFrame) -> pd.DataFrame:
    """Return closest DRep by |opinion_a - opinion_d| per delegator (tie by drep_id)."""
    a = Ae[['delegator_id','opinion']].rename(columns={'opinion':'op_a'}).copy()
    d = De[['drep_id','opinion']].rename(columns={'opinion':'op_d'}).copy()
    a['key'] = 1; d['key'] = 1
    pairs = a.merge(d, on='key').drop(columns=['key'])
    pairs['distance'] = (pairs['op_a'] - pairs['op_d']).abs()
    nearest = (pairs.sort_values(['delegator_id','distance','drep_id'])
                    .groupby('delegator_id', as_index=False)
                    .first())
    nearest = nearest.rename(columns={'op_d':'drep_opinion'})
    return nearest[['delegator_id','drep_id','drep_opinion','distance']]


In [16]:
delegator_rows = []
drep_rows = []
epochs = sorted(dreps['epoch'].unique())

# remember last assigned drep_id per delegator across epochs
prev_assignment = {}  # delegator_id (str) -> drep_id (str)

for e in epochs:
    D = dreps.loc[dreps['epoch'] == e, ['drep_id','opinion','stake']].copy()
    A = deleg.loc[deleg['epoch'] == e, ['delegator_id','opinion','stake','s']].copy()

    # make sure ids are strings for stable comparison
    D['drep_id'] = D['drep_id'].astype(str)
    A['delegator_id'] = A['delegator_id'].astype(str)

    nearest = assign_closest(A, D)  # returns columns: delegator_id, drep_id, drep_opinion, distance
    nearest['drep_id'] = nearest['drep_id'].astype(str)

    joined = A.merge(nearest, on='delegator_id', how='left')

    # Delegator rows (+ switched flag)
    for _, r in joined.iterrows():
        did = str(r['delegator_id'])
        drep_now = str(r['drep_id'])
        drep_prev = prev_assignment.get(did)
        switched = int(drep_prev is not None and drep_prev != drep_now)
        prev_assignment[did] = drep_now  # update for next epoch

        delegator_rows.append({
            'epoch': int(e),
            'delegator_id': did,
            'opinion': float(r['opinion']),
            'stake': float(r['stake']),
            's': float(r['s']),
            'drep_id': drep_now,
            'drep_opinion': float(r['drep_opinion']),
            'distance': float(r['distance']),
            'switched': switched,   # <-- NEW COLUMN
        })

    # DRep aggregates
    own = dict(zip(D['drep_id'], D['stake']))
    delegated_stake = joined.groupby('drep_id')['stake'].sum().to_dict()
    indeg = joined.groupby('drep_id')['delegator_id'].count().to_dict()
    avgdist = joined.groupby('drep_id')['distance'].mean().to_dict()

    total_Wprime = 0.0
    tmp = []
    for d_id, op in zip(D['drep_id'], D['opinion']):
        del_st = float(delegated_stake.get(d_id, 0.0))
        own_st = float(own.get(d_id, 0.0))
        Wp = own_st + del_st
        total_Wprime += Wp
        tmp.append({
            'epoch': int(e),
            'drep_id': d_id,
            'opinion': float(op),
            'stake': own_st,
            'delegated_stake': del_st,
            'indegree': int(indeg.get(d_id, 0)),
            'avg_distance': float(avgdist.get(d_id, 0.0)),
            'Wprime': Wp,
        })

    for row in tmp:
        row['Wprime_share'] = (row['Wprime'] / total_Wprime) if total_Wprime > 0 else 0.0
        drep_rows.append(row)

deleg_out = OUT_DIR / "delegators_state_with_delegations.csv"
dreps_out = OUT_DIR / "dreps_state_with_wprime.csv"

pd.DataFrame(delegator_rows).to_csv(deleg_out, index=False)
pd.DataFrame(drep_rows).to_csv(dreps_out, index=False)

print("Saved:", deleg_out.resolve())
print("Saved:", dreps_out.resolve())


Saved: /Users/Joel/Documents/GitHub/ada_drep/simulation/csv_out_optimal/delegators_state_with_delegations.csv
Saved: /Users/Joel/Documents/GitHub/ada_drep/simulation/csv_out_optimal/dreps_state_with_wprime.csv
